# DS6021 Final Project
## Emmett Hannam, Jarrett Markman, Weston Williams, Jeffrey Zhang

### Data Cleaning

In [ ]:
# Import packages and libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor  
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet, LinearRegression
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import silhouette_score, mean_squared_error
from umap import UMAP
import umap.umap_ as umap

In [ ]:
# Build function to read in data based on input/output for a given week
def read_data(prefix, wk):
    # Set "file" equal to the inputted file path and week
    file = f"{prefix}_2023_w{wk}.csv"
    return pd.read_csv(file) # Return the file read
# Load all input data for a given week
def load_weekly_data(weeks=range(1, 19)): # For weeks 1-18
    # Create empty data frames
    input_data = pd.DataFrame()

    # Iterate through weeks
    for wk in weeks:
        # Set week as a string (e.g. 1 becomes "01")
        wk_str = f"{wk:02d}"

        # Read the data 
        input_df = read_data("train/input", wk_str)

        # Concatenate the data for each week
        input_data = pd.concat([input_data, input_df], ignore_index=True)

    return input_data
input_data = load_weekly_data()

In [ ]:
# Clean tracking data
# Clean orientation and direction
input_data["o_clean"] = (-(input_data["o"] - 90)) % 360
input_data["dir_clean"] = (-(input_data["dir"] - 90)) % 360
# Set x on same scale based on play direction
input_data["x_clean"] = np.where(
      input_data["play_direction"] == "left",
      120 - input_data["x"],
      input_data[
          "x"
      ], 
  )
# y, s, a already clean
input_data["y_clean"] = input_data["y"]
input_data["s_clean"] = input_data["s"]
input_data["a_clean"] = input_data["a"]
# Clean orientation based on play direction
input_data["o_clean"] = np.where(
    input_data["play_direction"] == "left", 180 - input_data["o_clean"], input_data["o_clean"]
)
# Clean orientation, direction, vx, and vy
input_data["o_clean"] = (input_data["o_clean"] + 360) % 360 
input_data["dir_clean"] = (input_data["dir_clean"] + 360) % 360
input_data["dir_radians"] = np.radians(input_data["dir_clean"])
input_data["v_x"] = input_data["s_clean"] * np.cos(input_data["dir_radians"])
input_data["v_y"] = input_data["s_clean"] * np.sin(input_data["dir_radians"])
# Sort data based on nfl, game, play and frame id
input_data = input_data.sort_values(by=['nfl_id', 'game_id', 'play_id', 'frame_id'])
# Group by the specified columns and apply the shift within each group
input_data["prev_x"] = input_data.groupby(['nfl_id', 'game_id', 'play_id'])["x_clean"].shift(1)
input_data["prev_y"] = input_data.groupby(['nfl_id', 'game_id', 'play_id'])["y_clean"].shift(1)
# Remove the nans with no prev_x and prev_y
input_data = input_data.dropna(subset=["prev_x", "prev_y"])
# Remove features from input_data
df = input_data.drop(columns=['absolute_yardline_number', 'player_name', 'player_height', 'player_weight', 'player_birth_date', 'player_position', 'wk',
                             'x', 'y', 's', 'a', 'o', 'dir', 'play_direction', 'num_frames_output'])
# Sample 100000 rows
#df_sample = df.sample(n=100000, random_state=42)
# Output to csv
#df_sample.to_csv("input_data_clean.csv", index=False)

### EDA and Data Analysis

In [ ]:
# Look at all columns in df
df.columns

In [ ]:
# Summarize numeric features
numeric = ["x_clean", "y_clean", "s_clean", "a_clean", 
                "o_clean", "dir_clean", "v_x", "v_y"]
desc = df[numeric].describe().T
print(desc)

In [ ]:
# Plot speed and acceleration distributions
plt.figure(figsize=(14,6))

plt.subplot(1,2,1)
sns.histplot(df["s_clean"], bins=40, kde=True)
plt.title("Distribution of Player Speed")
plt.xlabel("Speed (yards/sec)")

plt.subplot(1,2,2)
sns.histplot(df["a_clean"], bins=40, kde=True, color="orange")
plt.title("Distribution of Player Acceleration")
plt.xlabel("Acceleration (yards/sec²)")

plt.tight_layout()
plt.show()

In [ ]:
# Speed and acceleration boxplots by position
plt.figure(figsize=(16,6))

plt.subplot(1,2,1)
sns.boxplot(data=df, x="player_position", y="s_clean")
plt.title("Speed by Position")

plt.subplot(1,2,2)
sns.boxplot(data=df, x="player_position", y="a_clean")
plt.title("Acceleration by Position")

plt.tight_layout()
plt.show()

In [ ]:
# Look at speed over time for all players in a sample game and play
sample_play = df[df["play_id"] == df["play_id"].iloc[0]]
sample_game = sample_play[sample_play["game_id"] == sample_play["game_id"].iloc[0]]

plt.figure(figsize=(10,5))
sns.lineplot(
    data=sample_play,
    x="frame_id",
    y="s_clean",
    hue="nfl_id",
    legend=False
)
plt.title("Speed Over Time for All Players During Sample game")
plt.xlabel("Frame ID")
plt.ylabel("Speed")
plt.show()

In [ ]:
# Look at velocity vectors for a specific play
play_id = df["play_id"].iloc[101]  

# specific play AND frame 1
play_frame = df[(df["play_id"] == play_id) & (df["frame_id"] == 1)].copy()

plt.figure(figsize=(12, 5))

plt.quiver(
    play_frame["x_clean"],
    play_frame["y_clean"],
    play_frame["v_x"],
    play_frame["v_y"],
    angles='xy',
    scale_units='xy',
    scale=1
)

plt.title(f"Velocity Vectors for Play {play_id} (Frame 1)")
plt.xlabel("X Position (yds)")
plt.ylabel("Y Position (yds)")
plt.xlim(0, 120)
plt.ylim(0, 53.3)

plt.show()

In [ ]:
# Look at unique yardline numbers over a game
gameID = df["game_id"].iloc[101]
playID = df["play_id"].iloc[101]
df2 = df[(df["game_id"] == gameID) & (df["player_name"] == 'Jared Goff')]
df2['absolute_yardline_number'].unique()

In [ ]:
# Look at the data for a random game_id
playIDRavens = df[(df['game_id'] == 2023123000)]
playIDRavens

### KMeans, PCA, PCR

In [ ]:
# Previewing data for later preparation 
df.head()

In [ ]:
# Look at shape
df.shape

In [ ]:
# Get value counts for player_side
df["player_side"].value_counts()

In [ ]:
# Look at columns in df
df.columns

#### PCA

In [ ]:
# Filter for only offensive data
offense = df[df['player_side'] == 'Offense'].copy()
# get pre snap data only
pre_snap = offense.loc[offense.groupby(['game_id', 'play_id', 'nfl_id'])['frame_id'].idxmin()]

feature_columns = ["x_clean", "y_clean", "s_clean", "a_clean", "o_clean", "dir_clean", "v_x", "v_y"]

X = pre_snap[feature_columns].values # Subset for predictors of interest above 

offense.head()

In [ ]:
pipe = Pipeline([ # pipeline definition, with only PCA to start of 
    ('scaler', StandardScaler()),
    ('pca', PCA())
])

pipe.fit(X) # fit the pipeline to data

pca_model = pipe.named_steps["pca"]
explained_var = pca_model.explained_variance_ratio_
cum_explained_var = np.cumsum(explained_var)
n_components = len(explained_var)

ev_df = pd.DataFrame({ # PCA metrics 
    "PC": np.arange(1, n_components + 1),
    "ExplainedVariance": explained_var,
    "CumulativeVariance": cum_explained_var
})


fig_scree = px.line( # plotly scree plot HERE
    ev_df, x="PC", y="ExplainedVariance",
    markers=True,
    title="Scree Plot: Proportion of Variance Explained"
)
fig_scree.show()

In [ ]:
ev_df

In [ ]:
X_pca_scores = pipe.transform(X)
pc_cols = [f"PC{i}" for i in range(1, n_components + 1)] 
# check transformed data and associated PC columns
scores_df = pd.DataFrame(X_pca_scores, columns=pc_cols)
scores_df

In [ ]:
# transforming data to check for optimal amount of clusters
silhouette = {}
inertias = {}

X_pca = pipe.transform(X) 

for k in range(2, 15): # check for optimal k-values
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(X_pca)

    silhouette[k] = silhouette_score(X_pca, kmeans.labels_)
    inertias[k] = kmeans.inertia_

In [ ]:
# Identify best k-value here based on silhouette score
optimal_k = max(silhouette, key = silhouette.get)

print("Best k based on Silhouette Score:", optimal_k)

#### KMeans

In [ ]:
# add kmeans in here, choosing k = 8, to try to get variability in the groups, before choosing the optimal number of clusters
# Redefining the FULL pipeline here with K-Means included

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),  
    ('kmeans', KMeans(n_clusters=8, n_init=10, random_state=42)) # 3,4 clusters, probbaly better, but want more to see more variabiltiy in different types of plays
])

pipe.fit(X)

pca = pipe.named_steps['pca']

labels = pipe.named_steps['kmeans'].labels_
pre_snap["cluster"] = labels

# stats, checking cluster summaries 

print("PCA Components:", pca.n_components_)
print("Explained variance (ratio):", pca.explained_variance_ratio_)
print("Cluster counts:\n", pre_snap["cluster"].value_counts())

explained_var = pca.explained_variance_ratio_
cum_explained_var = np.cumsum(explained_var)
n_components = len(explained_var)

df_pca = pd.DataFrame({
    "PC": np.arange(1, n_components + 1),
    "ExplainedVariance": explained_var,
    "CumulativeVariance": cum_explained_var
})

fig_scree = px.line(
    df_pca, x="PC", y="ExplainedVariance",
    markers=True,
    title="Scree Plot: Proportion of Variance Explained"
)
fig_scree.show()

In [ ]:
X_pca_scores = pipe.transform(X)
pc_cols = [f"PC{i}" for i in range(1, n_components + 1)]

scores_df = pd.DataFrame(X_pca_scores, columns=pc_cols) # PCA influenec by cluster
scores_df

In [ ]:
plt.figure(figsize=(8, 6)) # visualizing K-Means clusters in PCA space
sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=labels
)
plt.title("K-Means Clusters Visualized in PCA Space")
plt.ylabel("PC2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

In [ ]:
cluster_summary = pre_snap.groupby("cluster")[feature_columns].mean()
cluster_summary

In [ ]:
# this section was used to calculate silhouette scores for different k-values, but incorrect because it used original data, not PCA transformed data

#from sklearn.metrics import silhouette_score

#sil_scores = []
#K_values_sil = list(range(2, 11))

#for k in K_values_sil:
#    pipe.set_params(kmeans__n_clusters=k)
#    pipe.fit(X)

#    labels = pipe["kmeans"].labels_

#    sil = silhouette_score(X, labels)

#   sil_scores.append(sil)
#sil_scores

In [ ]:
# plotting silhouette scores for different k-values
lines = list(range(2, 11))
fig = px.line(
    x=lines,
    y=silhouette,
    markers= True,
    title="Silhouette Scores",
    labels={"x": "Number of K Clusters", "y": "Silhouette Scores"}
)

fig.show()

In [ ]:
# plotting elbow scores for different k-values

K_values = list(range(1, 11))
wcss = []

for k in K_values: # need within cluster sum of squares calculation HERE
    pipe.set_params(kmeans__n_clusters=k)
    pipe.fit(X)
    inertia = pipe["kmeans"].inertia_
    wcss.append(inertia)


fig = px.line( # plotting elbow plot for additional cluster evaluation
    x=K_values,
    y=wcss,
    markers=True,
    title="Elbow Plot",
    labels={"x": "Number of K Clusters", 
            "y": "WCSS"}
)

fig.show()

In [ ]:
wcss

#### KMeans, with $K$ = 4 as the optimal number of clusters

In [ ]:
pipe.set_params(kmeans__n_clusters=4) # setting K = 4
pipe.fit(X)

labels = pipe.named_steps['kmeans'].labels_
pre_snap["clusterk4"] = labels

X_pca = pipe.named_steps['pca'].transform(pipe.named_steps['scaler'].transform(X))

plt.figure(figsize=(10, 8)) # visualizing K-Means clusters in PCA space for K = 4
sns.scatterplot(x = X_pca[:, 0], y =X_pca[:, 1], hue=labels)
plt.title("K-Means for 4 Clusters in PCA Space")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cluster")
plt.show()

In [ ]:
cluster3_summary = pre_snap.groupby("clusterk4")[feature_columns].mean()
print(cluster3_summary)

#### PCR

In [ ]:
features = ['prev_x', 'prev_y', 'o_clean', 'dir_radians', 's_clean', 'a_clean', 'v_x', 'v_y']
X = offense[features].values
y = offense['x_clean'].values

X_train, X_test, y_train, y_test = train_test_split( # Standard train-test split
    X, y, test_size=0.25, random_state=42
)

pcr_pipe = Pipeline([ # FULL PCR pipeline
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=4)),
    ("linreg", LinearRegression())
])

pcr_pipe.fit(X_train, y_train)# fitting the PCR model on the training data

In [ ]:
# Get train/test R^2 and predictions
train_r2 = pcr_pipe.score(X_train, y_train)
test_r2  = pcr_pipe.score(X_test, y_test)


y_train_pred = pcr_pipe.predict(X_train)
y_test_pred  = pcr_pipe.predict(X_test)


In [ ]:
# Get train/test RMSE
train_rmse = np.sqrt(np.mean((y_train - y_train_pred)**2))
test_rmse  = np.sqrt(np.mean((y_test - y_test_pred)**2))
# Print out model metrics
print(f"Using n_components = {4}")
print(f"Train R²  : {train_r2:.4f}")
print(f"Test  R²  : {test_r2:.4f}")
print(f"Train RMSE: {train_rmse:.4f}")
print(f"Test  RMSE: {test_rmse:.4f}")

In [ ]:
# Get explained variance for PCA
pca_model = pcr_pipe.named_steps['pca']
print(f"\nExplained variance by 3 PCs: {pca_model.explained_variance_ratio_.sum():.4f}")
print(f"Single PC variance: {pca_model.explained_variance_ratio_}")

### Linear Regression

#### Goal 1: Measure the causal relationship between the tracking data and player coordinates

In measuring the causal relationship between the current player location, we chose to include:
- `prev_{loc}` (The previous x or y coordinate location)
- `o_clean` (Player orientation)
- `dir_radians` (The angle of player motion)
- `s_clean` (Speed in yards/second)
- `a_clean` (Acceleration in yards/second squared)
- `v_x` (Velocity along the x-axis)
- `v_y` (Velocity along the y-axis)

We can expect this predictors to collectively impact player movement, as they represent where they last were, where they are angled, and how and where they are moving. 

In [ ]:
# Use stats models to build a linear regression model for x_clean
# Set predictors and response
X_x = sm.add_constant(df[['prev_x', 'o_clean', 'dir_radians', 's_clean', 'a_clean', 'v_x', 'v_y']]) 
y_x = df[['x_clean']]
x_model = sm.OLS(y_x, X_x).fit()
print(x_model.summary())

In [ ]:
# Use stats models to build a linear regression model for y_clean
X_y = sm.add_constant(input_data[['prev_y', 'o_clean', 'dir_radians', 's_clean', 'a_clean', 'v_y', 'v_x']]) 
y_y = input_data[['y_clean']]
y_model = sm.OLS(y_y, X_y).fit() 
print(y_model.summary())

#### Test model assumptions (linearity, independence, vif, normality, heteroscedasticity)

In [ ]:
# Build a residuals/fitted plot to look at potential linearity and heteroscedasticity present
# Set plot parameters
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
# Set fitted values/residuals for x and y
fitted_vals_x = x_model.fittedvalues
residuals_x = x_model.resid
fitted_vals_y = y_model.fittedvalues
residuals_y = y_model.resid
# Build residuals/fitted plot for x and y coordinates
axes[0].scatter(fitted_vals_x, residuals_x, alpha=0.5)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel("Fitted values (x_model)")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals vs Fitted (x_model)")
axes[1].scatter(fitted_vals_y, residuals_y, alpha=0.5)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel("Fitted values (y_model)")
axes[1].set_title("Residuals vs Fitted (y_model)")
plt.tight_layout()
plt.show()

The residuals/fitted plot is a strong indication of *heteroscedasticity* present in the model, as the variance of errors is clearly not equal across fitted values. Additionally, the rhombus shape can be indicative of a non-linear presence within the model, and the fact that it's not truly capturing the relationship between the predictors and response. To combat these two issues, you can potentially perform various variable transformations among the response or predictor variables, which can include log terms, the inclusion of interaction terms polynomial terms, or removing variables from the model. 

In [ ]:
# VIF
def compute_vif(X):  
    # Drop intercept if present
    X_no_const = X.drop(columns=["const"], errors="ignore")
    
    vif_data = pd.DataFrame()
    vif_data["feature"] = X_no_const.columns
    vif_data["VIF"] = [
        variance_inflation_factor(X_no_const.values, i)
        for i in range(X_no_const.shape[1])
    ]
    return vif_data
print(compute_vif(X_x), compute_vif(X_y))

There doesn't appear to be any multicolinearity present within the model - though the VIF for `prev_y` in the y-coordinate model has a VIF > 5 which is a cause for concern. `prev_y` carries too much value in the model as it's representative of where the player last was, so it doesn't make sense to remove it. 

In [ ]:
# Test for normality of residuals using QQ plot for the x and y coordinate models
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sm.qqplot(x_model.resid, line='45', ax=axes[0])
axes[0].set_title("QQ Plot of Residuals (x_model)")
sm.qqplot(y_model.resid, line='45', ax=axes[1])
axes[1].set_title("QQ Plot of Residuals (y_model)")
plt.tight_layout()
plt.show()

In the qqplot, we can see that the data falls almost completely verticl and does not suggest that the sample distribution is any similar to the theoretical distribution. Given the distribution for theoretical quantiles hovers around 0, it's indicative that the data is extremely similar and clustered together as there isn't all too much different within the data. The normality assumption is violated.

Given the fact that the assumptions for linearity, heteroscedasticity, and normality appear to be violated (independence can be identified based on the *Durbin-Watson* value), we cannot interpret anything too much within this model. While features carry statistical significance and the model $R^2$'s are high, the assumptions being violated are indicative that the model results are unreliable, as they can lead to biased estimates and inaccurate predictions.

#### Goal 2: Attempt to predict player coordinates

While OLS can be effective in measuring the causal relationship between predictor variables with a given response, **Ridge** and **Lasso** regression can be effective in predicting a given response variable based on select predictors. **ElasticNet** is the best of both worlds, as it combines the coefficient shrinkage that *Ridge* performs, and also specific feature selection, like *Lasso*. Given the goal is to maximize predictive accuracy, we'll include the previous X and Y location in the model. 

In [ ]:
def build_model(target):
    if target == "x_clean":
        # Set numerical and categorical features
        nums = ['prev_x', 'o_clean', 'dir_radians', 'y_clean', 's_clean', 'a_clean', 'v_x', 'v_y']
        cats = ['player_side', 'player_role']
        X = input_data[nums + cats]
        y = input_data[['x_clean']]
    elif target == "y_clean":
        nums = ['prev_y', 'o_clean', 'dir_radians', 'x_clean', 's_clean', 'a_clean', 'v_x', 'v_y']
        cats = ['player_side', 'player_role']
        X = input_data[nums + cats]
        y = input_data[['y_clean']]
    return nums, cats, X, y
def fit_elastic_net(target, alphas=[0.1, 0.3, 0.5, 0.7, 0.8, 0.9], l1_ratio=0.5, test_size=0.2, random_state=22903):
    # Build the data
    nums, cats, X, y = build_model(target)
    y = y.values.ravel()  # flatten to 1D
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    
    # Preprocessing
    preprocess = ColumnTransformer([
        ("num", StandardScaler(), nums),
        ("cat", OneHotEncoder(), cats)
    ])
    
    best_alpha = None
    best_score = float("inf")
    best_model = None
    
    for alpha in alphas:
        model = Pipeline([
            ("preprocess", preprocess),
            ("elasticnet", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=1000, random_state=random_state))
        ])
        model.fit(X_train, y_train)
        preds = model.predict(X_test)  # evaluate on test set
        score = mean_squared_error(y_test, preds)
        
        if score < best_score:
            best_score = score
            best_alpha = alpha
            best_model = model
    
    print(f"Best alpha for {target}: {best_alpha}")
    print(f"Test RMSE: {np.sqrt(best_score):.4f}")
    return best_model, X_test, y_test

In [ ]:
enet_x_model, X_test_x, y_test_x = fit_elastic_net("x_clean")
enet_y_model, X_test_y, y_test_y = fit_elastic_net("y_clean")

In [ ]:
y_pred_x = enet_x_model.predict(X_test_x)
y_pred_y = enet_y_model.predict(X_test_y)

# Create a figure with 2 subplots, side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(y_test_x, y_pred_x, alpha=0.5)
axes[0].plot(
    [y_test_x.min(), y_test_x.max()],
    [y_test_x.min(), y_test_x.max()],
    color="red", linestyle="--"
)
axes[0].set_title("X_clean: Actual vs Predicted (Test Set)")
axes[0].set_xlabel("Actual x_clean")
axes[0].set_ylabel("Predicted x_clean")
axes[0].grid(True)

axes[1].scatter(y_test_y, y_pred_y, alpha=0.5)
axes[1].plot(
    [y_test_y.min(), y_test_y.max()],
    [y_test_y.min(), y_test_y.max()],
    color="red", linestyle="--"
)
axes[1].set_title("Y_clean: Actual vs Predicted (Test Set)")
axes[1].set_xlabel("Actual y_clean")
axes[1].set_ylabel("Predicted y_clean")
axes[1].grid(True)

plt.tight_layout()
plt.show()

Based on the scatterplots above, we can see that the elastic net model was effective in predicting x locations, but struggled a little with predicting y locations. 

In [ ]:
def get_elasticnet_coefficients(model, nums, cats):
    """
    Extract coefficients from a fitted ElasticNet pipeline and return as a DataFrame.
    """
    # Preprocessor and model
    preprocessor = model.named_steps['preprocess']
    enet = model.named_steps['elasticnet']
    
    # Get feature names from preprocessing
    cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(cats)
    feature_names = np.concatenate([nums, cat_features])
    
    # Get coefficients
    coefs = enet.coef_
    
    # Combine into DataFrame
    coef_df = pd.DataFrame({
        'feature': feature_names,
        'coefficient': coefs
    }).sort_values(by='coefficient', key=abs, ascending=False)  # sort by magnitude
    
    return coef_df
# Get nums and cats for x_clean
nums_x, cats_x, _, _ = build_model("x_clean")
coef_x = get_elasticnet_coefficients(enet_x_model, nums_x, cats_x)
print("Elastic Net coefficients for x_clean:")
print(coef_x)
# Get nums and cats for y_clean
nums_y, cats_y, _, _ = build_model("y_clean")
coef_y = get_elasticnet_coefficients(enet_y_model, nums_y, cats_y)
print("\nElastic Net coefficients for y_clean:")
print(coef_y)

In looking at the coefficients for both the X and Y coordinate models, we can see the features being used to make these predictions. For the X-coordinate model, the previous X location as well as speed and accelerationhave a strong effect on current X, as well as the side of the ball and when a player is in defensive coverage. 

For the y-coordinate model the effects appear to come from the previous Y location, as well as the Y velocity, along with the orientation and the direction. 

In [ ]:
# Place functions built to run streamlit app
# Create build model function for building data for a linear regression model
def build_model(input_data, target):
    """
    Input: input_data, target

    Description: Build the data for a linear regression model

    Output: nums, cats, X, y
    """
    if target == "x_clean": # If the target is x_clean
        # Set numerical and categorical features
        nums = ['prev_x', 'o_clean', 'dir_radians', 'y_clean', 's_clean', 'a_clean', 'v_x', 'v_y']
        cats = ['player_side', 'player_role']
        X = input_data[nums + cats]
        y = input_data[['x_clean']]
    elif target == "y_clean": # If the target is y_clean
        nums = ['prev_y', 'o_clean', 'dir_radians', 'x_clean', 's_clean', 'a_clean', 'v_x', 'v_y']
        cats = ['player_side', 'player_role']
        X = input_data[nums + cats]
        y = input_data[['y_clean']]
    else: # If the target is not x_clean or y_clean
        raise ValueError("Invalid target. Must be 'x_clean' or 'y_clean'.")
    return nums, cats, X, y

# Function to fit an elastic net model based on parameters
def fit_elastic_net(input_data, target, alpha=0.1, l1_ratio=0.5, test_size=0.2):
    """
    Input: input_data, target, alpha, l1_ratio, test_size

    Description: Fit an elastic net model based on parameters

    Output: model, X_test, y_test, preds, nums, cats, rmse
    """
    # Build the data
    nums, cats, X, y = build_model(input_data, target)
    y = y.values.ravel()  # Flatten to 1D
    
    # Train/test split based on test size 
    X_train, X_test, y_train, y_test = train_test_split(
        # Split x, y, input test_size parameter
        X, y, test_size=test_size, random_state=22903
    )
    
    # Set model preprocessing with standard scaler and one hot encoder for numerical and categorical features
    preprocess = ColumnTransformer([
        ("num", StandardScaler(), nums),
        ("cat", OneHotEncoder(), cats)
    ])

    # Use preprocessing and elastic net to set the model pipeline
    model = Pipeline([
        ("preprocess", preprocess),
        # Build elastic net model based on given alpha and l1_ratio
        ("elasticnet", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=1000, random_state=22903))
    ])
    # Fit the model, make predictions, and get model RMSE
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    return model, X_test, y_test, preds, nums, cats, rmse

# Build function to get the elastic net coefficients for a given model
def get_elasticnet_coefficients(model, nums, cats):
    """
    Extract coefficients from a fitted ElasticNet pipeline and return as a DataFrame.
    """
    # Preprocessor and model
    # Get preprocessor and elastic net for model
    preprocessor = model.named_steps['preprocess']
    enet = model.named_steps['elasticnet']
    
    # Get feature names from preprocessing
    cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(cats)
    feature_names = np.concatenate([nums, cat_features])
    
    # Get coefficients
    coefs = enet.coef_
    
    # Combine into DataFrame
    coef_df = pd.DataFrame({
        'feature': feature_names,
        'coefficient': coefs
    }).sort_values(by='coefficient', key=abs, ascending=False)  # sort by magnitude
    
    return coef_df

### Logistic Regression

### KNN

### TSNE, UMap with hyperparameter tuning

#### Sources

https://plotly.com/python/t-sne-and-umap-projections/ 

https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html 

https://umap-learn.readthedocs.io/en/latest/basic_usage.html 

In [ ]:
# data prepraation
positions = pre_snap["player_position"].values # assign labels 
roles = pre_snap["player_role"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# take subset of data HERE because t-sne and umap are slow

n_sample = min(5000, len(X_scaled)) 
sample_idx = np.random.choice(len(X_scaled), n_sample, replace=False)
X_sample = X_scaled[sample_idx]
positions_sample = positions[sample_idx] # assign labels
roles_sample = roles[sample_idx] # assign labels

# initialize and fit t-SNE and UMAP

 # reducing TNSE to 2D, the parameter being tuned
    # Need to initialize t-SNE with PCA here 
    # lets t-SNE choose an appropriate learning rate

tsne = TSNE(
    n_components=2, 
    init="pca", 
    learning_rate="auto", 
    random_state=42
).fit_transform(X_sample)

umap_result = umap.UMAP(
    n_components=2, 
    random_state=42
).fit_transform(X_sample)

In [ ]:
# establish data frame of results for both t-SNE and UMAP
tsne_df = pd.DataFrame({
    'tsne1': tsne[:, 0],
    'tsne2': tsne[:, 1],
    'position': positions_sample,
    'role': roles_sample
})

umap_df = pd.DataFrame({
    'umap1': umap_result[:, 0],
    'umap2': umap_result[:, 1],
    'position': positions_sample,
    'role': roles_sample
})

In [ ]:
tsne_df.head()

In [ ]:
umap_df.head()

In [ ]:
pca = PCA(n_components=2, random_state=42).fit_transform(X_sample)
# want to compare with t-SNE and UMAP, so will reduce to 2 PCs for consistency and interpretability

for emb, title in zip([pca, tsne, umap_result], ["PCA", "t-SNE", "UMAP"]): # looping through all models to plot them 
    plt.figure(figsize=(10, 8))
    uniq = np.unique(positions_sample) # grab unique positions 

    for pos in uniq:
        mask = positions_sample == pos
        plt.scatter(emb[mask, 0], emb[mask, 1], s=20, alpha=0.8, label = pos) # define clustering for each position 

    # plot formatting and such 
    plt.xlabel("Dim 1")
    plt.ylabel("Dim 2")

    plt.title(title)

    plt.legend(title="Position")
    plt.show()

In [ ]:
# Hyperparameter Tuning 
# Tuning perplexity for t-SNE

def scatter_by_position(emb, title):
    plt.figure(figsize=(10, 8))
    for pos in np.unique(positions_sample):
        m = positions_sample == pos # position selection here
        plt.scatter(emb[m, 0], emb[m, 1], s=18, label=pos)
    
    # plot formatting
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel("Dim 1", fontsize=12)
    plt.ylabel("Dim 2", fontsize=12)
    plt.legend(title = "Position")

    plt.show()

# Parameter to tune for t-SNE: perplexity
for perp in [5, 30, 50]:
    emb = TSNE(n_components=2, perplexity=perp, init="pca", learning_rate="auto", random_state=42).fit_transform(X_sample)

    # reducing TNSE to 2D, the parameter being tuned
    # Need to initialize t-SNE with PCA here 
    # lets t-SNE choose an appropriate learning rate

    scatter_by_position(emb, f"t-SNE (perplexity={perp})")

In [ ]:
# Hyperparamter Tuning: tuning n_neighbors & min_dist for umap

for nn in [5, 15, 50]: # testing various neighbor sizes
    for md in [0.0, 0.5]: # various tightness within the cluster itself 
        emb = umap.UMAP(n_components=2, n_neighbors=nn, min_dist=md, random_state=42).fit_transform(X_sample)
        scatter_by_position(emb, f"UMAP (n_neighbors={nn}, min_dist={md})")

### Streamlit App